# Applicant Journey Flow

This notebook recreates the structure in `flow-chart-draft.pdf` as an interactive Altair flow chart. Use the dropdowns to filter by application year, school, and grade level, then hover on nodes, transitions, and bars to inspect counts.


In [ ]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

alt.renderers.enable("html")
alt.data_transformers.disable_max_rows()

pd.set_option("display.max_columns", None)

DATA_PATH = Path("data") / "Application History v3.csv"
DATA_PATH


In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

date_cols = [
    col
    for col in df.columns
    if col.endswith("DateDMID") or col in {"LastRunDMID", "SchStartDate", "SchEndDate"}
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col].astype("string"), format="%Y%m%d", errors="coerce")

if "AcceptedDate" in df.columns:
    df["AcceptedDate"] = pd.to_datetime(df["AcceptedDate"], errors="coerce")

accepted_flag = (
    df["WasAccepted"].eq(1)
    | df["AcceptedDateDMID"].notna()
    | df["AcceptedDate"].notna()
)
registered_flag = accepted_flag & (
    df["WasRegistered"].eq(1)
    | df["RegisteredDateDMID"].notna()
    | df["Enrolled"].eq(1)
    | df["EnrolledDateDMID"].notna()
)
enrolled_flag = registered_flag & (
    df["Enrolled"].eq(1)
    | df["EnrolledDateDMID"].notna()
)
attending_flag = enrolled_flag & (
    df["StartedInFirstWeek"].eq(1)
    | df["StillEnrolledOnStartDatePlus7"].eq(1)
    | df["EnrolledOnCntFreezeDate"].eq(1)
)
accepted_at_lottery_flag = df["AcceptedAtLottery"].eq(1)
waitlisted_flag = df["AcceptedAtLottery"].eq(0)

GRADE_LABEL_MAP = {"4K": "Pre-K", "Y 5's": "Pre-K"}
GRADE_ORDER = {"Pre-K": 0, "K": 1, **{str(level): level + 1 for level in range(1, 13)}}


def normalize_grade_label(value):
    if pd.isna(value):
        return "Unknown"

    label = str(value).strip()
    if not label:
        return "Unknown"

    return GRADE_LABEL_MAP.get(label, label)


def grade_sort_key(label):
    normalized = normalize_grade_label(label)
    if normalized in GRADE_ORDER:
        return (0, GRADE_ORDER[normalized], normalized)

    numeric = pd.to_numeric(normalized, errors="coerce")
    if pd.notna(numeric):
        return (0, int(numeric) + 1, normalized)

    if normalized == "Unknown":
        return (2, float("inf"), normalized)

    return (1, float("inf"), normalized)

analysis_df = pd.DataFrame({
    "applicant_id": df.index.astype(str),
    "year_label": df["AcademicYearAppliedFor"].astype("Int64").astype("string").fillna("Unknown"),
    "school_label": df["AbbreviatedName"].fillna("Unknown").astype(str),
    "grade_label": df["GradeAbbreviation"].map(normalize_grade_label),
    "accepted": accepted_flag,
    "registered": registered_flag,
    "enrolled": enrolled_flag,
    "attending": attending_flag,
    "accepted_at_lottery": accepted_at_lottery_flag,
    "waitlisted": waitlisted_flag,
})
analysis_df["final_outcome"] = analysis_df["attending"].map({True: "Attending", False: "Not Attending"})
analysis_df["full_path"] = "Apply"
analysis_df.loc[analysis_df["waitlisted"], "full_path"] += " > Waitlisted"
analysis_df.loc[analysis_df["accepted_at_lottery"], "full_path"] += " > Accepted"
analysis_df.loc[analysis_df["waitlisted"] & analysis_df["accepted"], "full_path"] += " > Accepted"
analysis_df.loc[analysis_df["registered"], "full_path"] += " > Register"
analysis_df.loc[analysis_df["enrolled"], "full_path"] += " > Enroll"
analysis_df.loc[analysis_df["attending"], "full_path"] += " > Attending"
analysis_df.loc[analysis_df["waitlisted"] & ~analysis_df["accepted"], "full_path"] += " > Not Attending"
analysis_df.loc[analysis_df["accepted"] & ~analysis_df["registered"], "full_path"] += " > Not Attending"
analysis_df.loc[analysis_df["registered"] & ~analysis_df["enrolled"], "full_path"] += " > Not Attending"
analysis_df.loc[analysis_df["enrolled"] & ~analysis_df["attending"], "full_path"] += " > Not Attending"

print(f"Loaded {len(df):,} application rows")
analysis_df[["year_label", "school_label", "grade_label", "full_path", "final_outcome"]].head()


## Assumptions

- `Apply` is every application row in the source CSV.
- `Apply -> Accepted` uses `AcceptedAtLottery = 1` as the immediate-offer branch from the draft.
- `Apply -> Waitlisted` uses `AcceptedAtLottery = 0`; applicants later accepted move from `Waitlisted -> Accepted`.
- `Accepted -> Register`, `Register -> Enroll`, and `Enroll -> Attending` are proxied from the registration, enrollment, and attendance indicators already present in the file.
- Every applicant ends in exactly one terminal node: `Attending` or `Not Attending`.
- Counts are exposed through hover tooltips rather than always-on labels so the chart stays legible.


In [ ]:
def event_rows(mask, key, value):
    return analysis_df.loc[mask, ["applicant_id", "year_label", "school_label", "grade_label"]].assign(**{key: value})


def path_event_rows(mask, key, value):
    return analysis_df.loc[
        mask,
        ["applicant_id", "year_label", "school_label", "grade_label", "full_path", "final_outcome"],
    ].assign(**{key: value})


def rollup_counts(events, key_cols):
    if isinstance(key_cols, str):
        key_cols = [key_cols]

    filter_cols = ["year_label", "school_label", "grade_label"]
    renamed_filter_cols = {
        "year_label": "year_option",
        "school_label": "school_option",
        "grade_label": "grade_option",
    }
    base = events[key_cols + filter_cols].copy()
    rolled = pd.concat(
        [
            base.assign(
                **{
                    col: "All" if include_all else base[col]
                    for col, include_all in zip(filter_cols, combo)
                }
            ).rename(columns=renamed_filter_cols)
            for combo in [(False, False, False), (True, False, False), (False, True, False), (False, False, True), (True, True, False), (True, False, True), (False, True, True), (True, True, True)]
        ],
        ignore_index=True,
    )
    return (
        rolled.groupby(["year_option", "school_option", "grade_option", *key_cols], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )


PATH_LABEL_MAP = {
    "Apply > Accepted > Not Attending": "Withdrew after Acceptance",
    "Apply > Accepted > Register > Enroll > Attending": "Enrolled",
    "Apply > Accepted > Register > Enroll > Not Attending": "No Show",
    "Apply > Accepted > Register > Not Attending": "Withdrew after Registration",
    "Apply > Waitlisted > Accepted > Not Attending": "Withdrew after Acceptance (Previously Waitlisted)",
    "Apply > Waitlisted > Accepted > Register > Enroll > Attending": "Enrolled (Previously Waitlisted)",
    "Apply > Waitlisted > Accepted > Register > Enroll > Not Attending": "No Show (Previously Waitlisted)",
    "Apply > Waitlisted > Accepted > Register > Not Attending": "Withdrew after Registration (Previously Waitlisted)",
    "Apply > Waitlisted > Not Attending": "Remained Waitlisted",
}


def readable_path(path_value):
    return path_value.replace("Apply > ", "").replace(" > ", " -> ")


node_events = pd.concat(
    [
        event_rows(analysis_df["applicant_id"].notna(), "node", "Apply"),
        event_rows(analysis_df["waitlisted"], "node", "Waitlisted"),
        event_rows(analysis_df["accepted"], "node", "Accepted"),
        event_rows(analysis_df["registered"], "node", "Register"),
        event_rows(analysis_df["enrolled"], "node", "Enroll"),
        event_rows(~analysis_df["attending"], "node", "Not Attending"),
        event_rows(analysis_df["attending"], "node", "Attending"),
    ],
    ignore_index=True,
)

edge_events = pd.concat(
    [
        event_rows(analysis_df["waitlisted"], "edge", "apply_waitlisted"),
        event_rows(analysis_df["accepted_at_lottery"], "edge", "apply_accepted"),
        event_rows(analysis_df["waitlisted"] & analysis_df["accepted"], "edge", "waitlisted_accepted"),
        event_rows(analysis_df["registered"], "edge", "accepted_register"),
        event_rows(analysis_df["enrolled"], "edge", "register_enroll"),
        event_rows(analysis_df["waitlisted"] & ~analysis_df["accepted"], "edge", "waitlisted_not_attending"),
        event_rows(analysis_df["accepted"] & ~analysis_df["registered"], "edge", "accepted_not_attending"),
        event_rows(analysis_df["registered"] & ~analysis_df["enrolled"], "edge", "register_not_attending"),
        event_rows(analysis_df["enrolled"] & ~analysis_df["attending"], "edge", "enroll_not_attending"),
        event_rows(analysis_df["attending"], "edge", "enroll_attending"),
    ],
    ignore_index=True,
)

node_path_events = pd.concat(
    [
        path_event_rows(analysis_df["applicant_id"].notna(), "node", "Apply"),
        path_event_rows(analysis_df["waitlisted"], "node", "Waitlisted"),
        path_event_rows(analysis_df["accepted"], "node", "Accepted"),
        path_event_rows(analysis_df["registered"], "node", "Register"),
        path_event_rows(analysis_df["enrolled"], "node", "Enroll"),
        path_event_rows(~analysis_df["attending"], "node", "Not Attending"),
        path_event_rows(analysis_df["attending"], "node", "Attending"),
    ],
    ignore_index=True,
).rename(columns={"full_path": "path"})

edge_path_events = pd.concat(
    [
        path_event_rows(analysis_df["waitlisted"], "edge", "apply_waitlisted"),
        path_event_rows(analysis_df["accepted_at_lottery"], "edge", "apply_accepted"),
        path_event_rows(analysis_df["waitlisted"] & analysis_df["accepted"], "edge", "waitlisted_accepted"),
        path_event_rows(analysis_df["registered"], "edge", "accepted_register"),
        path_event_rows(analysis_df["enrolled"], "edge", "register_enroll"),
        path_event_rows(analysis_df["waitlisted"] & ~analysis_df["accepted"], "edge", "waitlisted_not_attending"),
        path_event_rows(analysis_df["accepted"] & ~analysis_df["registered"], "edge", "accepted_not_attending"),
        path_event_rows(analysis_df["registered"] & ~analysis_df["enrolled"], "edge", "register_not_attending"),
        path_event_rows(analysis_df["enrolled"] & ~analysis_df["attending"], "edge", "enroll_not_attending"),
        path_event_rows(analysis_df["attending"], "edge", "enroll_attending"),
    ],
    ignore_index=True,
).rename(columns={"full_path": "path"})

node_template = pd.DataFrame(
    [
        {"node": "Apply", "title": "Apply", "x": 5.0, "y": 1.0, "fill": "#ffffff", "stroke": "#222222", "dash": "solid"},
        {"node": "Waitlisted", "title": "Waitlisted", "x": 2.0, "y": 3.0, "fill": "#ffffff", "stroke": "#222222", "dash": "solid"},
        {"node": "Accepted", "title": "Accepted", "x": 8.0, "y": 3.0, "fill": "#ffffff", "stroke": "#222222", "dash": "solid"},
        {"node": "Register", "title": "Register", "x": 8.0, "y": 5.0, "fill": "#ffffff", "stroke": "#222222", "dash": "solid"},
        {"node": "Enroll", "title": "Enroll", "x": 8.0, "y": 7.0, "fill": "#ffffff", "stroke": "#222222", "dash": "solid"},
        {"node": "Not Attending", "title": "Not Attending", "x": 2.0, "y": 9.0, "fill": "#FDE2E2", "stroke": "#C93C37", "dash": "dashed"},
        {"node": "Attending", "title": "Attending", "x": 8.0, "y": 9.0, "fill": "#DDF3E4", "stroke": "#2E8B57", "dash": "solid"},
    ]
)
node_template["half_width"] = 1.45
node_template["x0"] = node_template["x"] - node_template["half_width"]
node_template["x1"] = node_template["x"] + node_template["half_width"]
node_template["y0"] = node_template["y"] - 0.38
node_template["y1"] = node_template["y"] + 0.38

edge_template = pd.DataFrame(
    [
        {"edge": "apply_waitlisted", "label": "Waitlisted", "x": 5.0, "y": 1.0, "x2": 2.0, "y2": 3.0, "label_side": 1.0, "label_pad": 0.36, "stroke": "#4d4d4d", "label_color": "#4d4d4d", "line_style": "solid"},
        {"edge": "apply_accepted", "label": "Accepted", "x": 5.0, "y": 1.0, "x2": 8.0, "y2": 3.0, "label_side": -1.0, "label_pad": 0.36, "stroke": "#4d4d4d", "label_color": "#4d4d4d", "line_style": "solid"},
        {"edge": "waitlisted_accepted", "label": "Off Waitlist", "x": 2.0, "y": 3.0, "x2": 8.0, "y2": 3.0, "label_side": -1.0, "label_pad": 0.42, "stroke": "#4d4d4d", "label_color": "#4d4d4d", "line_style": "solid"},
        {"edge": "accepted_register", "label": "Register", "x": 8.0, "y": 3.0, "x2": 8.0, "y2": 5.0, "label_side": 1.0, "label_pad": 0.52, "stroke": "#4d4d4d", "label_color": "#4d4d4d", "line_style": "solid"},
        {"edge": "register_enroll", "label": "Enroll", "x": 8.0, "y": 5.0, "x2": 8.0, "y2": 7.0, "label_side": 1.0, "label_pad": 0.52, "stroke": "#4d4d4d", "label_color": "#4d4d4d", "line_style": "solid"},
        {"edge": "waitlisted_not_attending", "label": "Stay Waitlisted", "x": 2.0, "y": 3.0, "x2": 2.0, "y2": 9.0, "label_side": 1.0, "label_pad": 0.6, "stroke": "#C93C37", "label_color": "#C93C37", "line_style": "dotted"},
        {"edge": "accepted_not_attending", "label": "Withdraw", "x": 8.0, "y": 3.0, "x2": 2.0, "y2": 9.0, "label_side": -1.0, "label_pad": 0.42, "stroke": "#C93C37", "label_color": "#C93C37", "line_style": "dotted"},
        {"edge": "register_not_attending", "label": "Withdraw", "x": 8.0, "y": 5.0, "x2": 2.0, "y2": 9.0, "label_side": -1.0, "label_pad": 0.42, "stroke": "#C93C37", "label_color": "#C93C37", "line_style": "dotted"},
        {"edge": "enroll_not_attending", "label": "No Show", "x": 8.0, "y": 7.0, "x2": 2.0, "y2": 9.0, "label_side": -1.0, "label_pad": 0.42, "stroke": "#C93C37", "label_color": "#C93C37", "line_style": "dotted"},
        {"edge": "enroll_attending", "label": "Shows Up", "x": 8.0, "y": 7.0, "x2": 8.0, "y2": 9.0, "label_side": 1.0, "label_pad": 0.58, "stroke": "#2E8B57", "label_color": "#2E8B57", "line_style": "dashed"},
    ]
)
edge_template["mid_x"] = (edge_template["x"] + edge_template["x2"]) / 2
edge_template["mid_y"] = (edge_template["y"] + edge_template["y2"]) / 2
edge_template["dx"] = edge_template["x2"] - edge_template["x"]
edge_template["dy"] = edge_template["y2"] - edge_template["y"]
edge_template["line_length"] = np.hypot(edge_template["dx"], edge_template["dy"])
edge_template["label_x"] = edge_template["mid_x"] - edge_template["label_side"] * edge_template["dy"] / edge_template["line_length"] * edge_template["label_pad"]
edge_template["label_y"] = edge_template["mid_y"] + edge_template["label_side"] * edge_template["dx"] / edge_template["line_length"] * edge_template["label_pad"]
edge_template["line_weight"] = 1

year_options = ["All"] + sorted(analysis_df["year_label"].dropna().unique().tolist())
school_options = ["All"] + sorted(analysis_df["school_label"].dropna().unique().tolist())
grade_options = ["All"] + sorted(analysis_df["grade_label"].dropna().unique().tolist(), key=grade_sort_key)
filter_grid = pd.MultiIndex.from_product(
    [year_options, school_options, grade_options],
    names=["year_option", "school_option", "grade_option"],
).to_frame(index=False)

node_counts = (
    filter_grid.merge(node_template[["node"]], how="cross")
    .merge(rollup_counts(node_events, "node"), on=["year_option", "school_option", "grade_option", "node"], how="left")
    .merge(node_template, on="node", how="left")
)
node_counts["count"] = node_counts["count"].fillna(0).astype(int)

edge_counts = (
    filter_grid.merge(edge_template[["edge"]], how="cross")
    .merge(rollup_counts(edge_events, "edge"), on=["year_option", "school_option", "grade_option", "edge"], how="left")
    .merge(edge_template, on="edge", how="left")
)
edge_counts["count"] = edge_counts["count"].fillna(0).astype(int)
edge_counts["line_weight"] = edge_counts["count"].clip(lower=1)

node_path_counts = (
    filter_grid.merge(node_template[["node"]], how="cross")
    .merge(rollup_counts(node_path_events, ["path", "node"]), on=["year_option", "school_option", "grade_option", "node"], how="left")
    .merge(node_template, on="node", how="left")
)
node_path_counts["count"] = node_path_counts["count"].fillna(0).astype(int)

edge_path_counts = (
    filter_grid.merge(edge_template[["edge"]], how="cross")
    .merge(rollup_counts(edge_path_events, ["path", "edge"]), on=["year_option", "school_option", "grade_option", "edge"], how="left")
    .merge(edge_template, on="edge", how="left")
)
edge_path_counts["count"] = edge_path_counts["count"].fillna(0).astype(int)
edge_path_counts["line_weight"] = edge_path_counts["count"].clip(lower=1)

path_counts = rollup_counts(
    analysis_df[["full_path", "year_label", "school_label", "grade_label"]].rename(columns={"full_path": "path"}),
    "path",
).merge(
    analysis_df[["full_path", "final_outcome"]].drop_duplicates().rename(columns={"full_path": "path"}),
    on="path",
    how="left",
)
path_counts["path_label"] = path_counts["path"].map(PATH_LABEL_MAP).fillna(path_counts["path"].map(readable_path))
path_counts["outcome_dash"] = path_counts["final_outcome"].map({"Attending": "solid", "Not Attending": "dashed"})
path_counts["outcome_fill"] = path_counts["final_outcome"].map({"Attending": "#49A35A", "Not Attending": "#E15759"})
path_counts["outcome_stroke"] = path_counts["final_outcome"].map({"Attending": "#2E8B57", "Not Attending": "#C93C37"})

outcome_counts = rollup_counts(
    analysis_df[["final_outcome", "year_label", "school_label", "grade_label"]].rename(columns={"final_outcome": "outcome"}),
    "outcome",
)
outcome_counts["outcome_dash"] = outcome_counts["outcome"].map({"Attending": "solid", "Not Attending": "dashed"})
outcome_counts["outcome_fill"] = outcome_counts["outcome"].map({"Attending": "#49A35A", "Not Attending": "#E15759"})
outcome_counts["outcome_stroke"] = outcome_counts["outcome"].map({"Attending": "#2E8B57", "Not Attending": "#C93C37"})
outcome_counts["count_text"] = outcome_counts["count"].map("{:,}".format)

selected_outcome_counts = rollup_counts(
    analysis_df[["full_path", "final_outcome", "year_label", "school_label", "grade_label"]].rename(columns={"full_path": "path", "final_outcome": "outcome"}),
    ["path", "outcome"],
).merge(
    outcome_counts[["year_option", "school_option", "grade_option", "outcome", "count"]].rename(columns={"count": "outcome_total"}),
    on=["year_option", "school_option", "grade_option", "outcome"],
    how="left",
)
selected_outcome_counts["share"] = selected_outcome_counts["count"] / selected_outcome_counts["outcome_total"].where(selected_outcome_counts["outcome_total"].gt(0))
selected_outcome_counts["share"] = selected_outcome_counts["share"].fillna(0)
selected_outcome_counts["share_label"] = (selected_outcome_counts["share"] * 100).round(1).map(lambda value: f"{value:.1f}%")
selected_outcome_counts["outcome_dash"] = selected_outcome_counts["outcome"].map({"Attending": "solid", "Not Attending": "dashed"})
selected_outcome_counts["outcome_fill"] = selected_outcome_counts["outcome"].map({"Attending": "#1F7A3B", "Not Attending": "#B6252A"})
selected_outcome_counts["outcome_stroke"] = selected_outcome_counts["outcome"].map({"Attending": "#14572B", "Not Attending": "#7F1D1D"})

path_label_space = max(int(path_counts["count"].max() * 1.55), 120000)
path_chart_max = max(int(path_counts["count"].max() * 1.12), 1000)

print(node_counts.query("year_option == 'All' and school_option == 'All' and grade_option == 'All'")[["node", "count"]])


In [ ]:
year_pick = alt.param(
    name="year_pick",
    value="All",
    bind=alt.binding_select(options=year_options, name="Year: "),
)
school_pick = alt.param(
    name="school_pick",
    value="All",
    bind=alt.binding_select(options=school_options, name="School: "),
)
grade_pick = alt.param(
    name="grade_pick",
    value="All",
    bind=alt.binding_select(options=grade_options, name="Grade: "),
)
path_select = alt.selection_point(
    name="path_select",
    fields=["path"],
    on="click",
    clear="dblclick",
    empty="none",
)
filter_expr = "datum.year_option == year_pick && datum.school_option == school_pick && datum.grade_option == grade_pick"

dash_scale = alt.Scale(domain=["solid", "dashed", "dotted"], range=[[1, 0], [7, 4], [1, 5]])
selection_active_expr = "length(data('path_select_store'))"

edge_base = alt.Chart(edge_counts).transform_filter(filter_expr).transform_calculate(
    edge_opacity=f"{selection_active_expr} ? (datum.count > 0 ? 0.14 : 0.04) : (datum.count > 0 ? 0.45 : 0.14)",
    edge_label_opacity=f"{selection_active_expr} ? (datum.count > 0 ? 0.3 : 0.12) : (datum.count > 0 ? 1 : 0.45)",
)
edge_chart = edge_base.mark_rule(strokeCap="round").encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    x2="x2:Q",
    y=alt.Y("y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    y2="y2:Q",
    size=alt.Size("line_weight:Q", scale=alt.Scale(range=[1.5, 10]), legend=None),
    color=alt.Color("stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("line_style:N", scale=dash_scale, legend=None),
    opacity=alt.Opacity("edge_opacity:Q", legend=None),
    tooltip=[
        alt.Tooltip("label:N", title="Transition"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
    ],
)

selected_edge_chart = alt.Chart(edge_path_counts).transform_filter(filter_expr).transform_filter(path_select).mark_rule(strokeCap="round").encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    x2="x2:Q",
    y=alt.Y("y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    y2="y2:Q",
    size=alt.Size("line_weight:Q", scale=alt.Scale(range=[6, 17]), legend=None),
    color=alt.Color("stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("line_style:N", scale=dash_scale, legend=None),
    opacity=alt.condition("datum.count > 0", alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip("path:N", title="Selected path"),
        alt.Tooltip("label:N", title="Transition"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
    ],
)

edge_labels = edge_base.mark_text(
    fontSize=15,
    fontWeight="bold",
    align="center",
    baseline="middle",
).encode(
    x=alt.X("label_x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    y=alt.Y("label_y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    text="label:N",
    color=alt.Color("label_color:N", scale=None, legend=None),
    opacity=alt.Opacity("edge_label_opacity:Q", legend=None),
)

selected_edge_labels = alt.Chart(edge_path_counts).transform_filter(filter_expr).transform_filter(path_select).mark_text(
    fontSize=15,
    fontWeight="bold",
    align="center",
    baseline="middle",
).encode(
    x=alt.X("label_x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    y=alt.Y("label_y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    text="label:N",
    color=alt.Color("label_color:N", scale=None, legend=None),
    opacity=alt.condition("datum.count > 0", alt.value(1), alt.value(0)),
)

node_base = alt.Chart(node_counts).transform_filter(filter_expr).transform_calculate(
    node_box_opacity=f"{selection_active_expr} ? (datum.count > 0 ? 0.18 : 0.06) : (datum.count > 0 ? 0.58 : 0.22)",
    node_title_opacity=f"{selection_active_expr} ? (datum.count > 0 ? 0.35 : 0.12) : (datum.count > 0 ? 1 : 0.6)",
)
node_boxes = node_base.mark_rect(
    cornerRadius=18,
    strokeWidth=1.9,
).encode(
    x=alt.X("x0:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    x2="x1:Q",
    y=alt.Y("y0:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    y2="y1:Q",
    color=alt.Color("fill:N", scale=None, legend=None),
    stroke=alt.Color("stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("dash:N", scale=dash_scale, legend=None),
    opacity=alt.Opacity("node_box_opacity:Q", legend=None),
    tooltip=[
        alt.Tooltip("title:N", title="Node"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
    ],
)

selected_node_boxes = alt.Chart(node_path_counts).transform_filter(filter_expr).transform_filter(path_select).mark_rect(
    cornerRadius=18,
    strokeWidth=4.2,
).encode(
    x=alt.X("x0:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    x2="x1:Q",
    y=alt.Y("y0:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    y2="y1:Q",
    color=alt.value("#FFF9E8"),
    stroke=alt.Color("stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("dash:N", scale=dash_scale, legend=None),
    opacity=alt.condition("datum.count > 0", alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip("path:N", title="Selected path"),
        alt.Tooltip("title:N", title="Node"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
    ],
)

node_titles = node_base.mark_text(
    fontSize=16,
    fontWeight="bold",
    color="#273043",
    align="center",
    baseline="middle",
).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    y=alt.Y("y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    text="title:N",
    opacity=alt.Opacity("node_title_opacity:Q", legend=None),
)

selected_node_titles = alt.Chart(node_path_counts).transform_filter(filter_expr).transform_filter(path_select).mark_text(
    fontSize=16,
    fontWeight="bold",
    color="#111827",
    align="center",
    baseline="middle",
).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0.2, 9.8]), axis=None),
    y=alt.Y("y:Q", scale=alt.Scale(domain=[0.4, 9.7], reverse=True), axis=None),
    text="title:N",
    opacity=alt.condition("datum.count > 0", alt.value(1), alt.value(0)),
)

flow_chart = (
    edge_chart
    + selected_edge_chart
    + edge_labels
    + selected_edge_labels
    + node_boxes
    + selected_node_boxes
    + node_titles
    + selected_node_titles
).properties(
    width=790,
    height=570,
    title="Applicant Journey Flow",
)

path_base = alt.Chart(path_counts).transform_filter(filter_expr).transform_filter(
    "datum.count > 0"
).transform_window(
    rank="rank(count)",
    sort=[alt.SortField("count", order="descending")],
).transform_filter("datum.rank <= 8").transform_calculate(
    path_id="'P' + format(datum.rank, '.0f')",
    label_start=f"{-path_label_space}",
    label_end="0",
    label_anchor=f"{-path_label_space + 2000}",
    count_text="format(datum.count, ',')"
)

path_click_target = path_base.mark_bar(opacity=0.001, cursor="pointer").encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("label_start:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=alt.Axis(title="Applicants")),
    x2=alt.datum(path_chart_max),
).add_params(path_select)

path_label_bg = path_base.mark_bar(
    color="white",
    opacity=0.97,
).encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("label_start:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=alt.Axis(title="Applicants")),
    x2="label_end:Q",
    opacity=alt.condition(path_select, alt.value(0.99), alt.value(0.86), empty=True),
    stroke=alt.condition(path_select, alt.value("#111827"), alt.value("#d7dde4")),
    strokeWidth=alt.condition(path_select, alt.value(2.4), alt.value(0.8)),
)

path_bars = path_base.mark_bar(
    cornerRadiusEnd=4,
    fillOpacity=0.84,
).encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("count:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=alt.Axis(title="Applicants")),
    x2=alt.datum(0),
    color=alt.Color("outcome_fill:N", scale=None, legend=None),
    stroke=alt.Color("outcome_stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("outcome_dash:N", scale=dash_scale, legend=None),
    strokeWidth=alt.condition(path_select, alt.value(4), alt.value(2.2)),
    opacity=alt.condition(path_select, alt.value(1), alt.value(0.24), empty=True),
    tooltip=[
        alt.Tooltip("path:N", title="Path"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
        alt.Tooltip("final_outcome:N", title="Outcome"),
    ],
)

path_id_labels = path_base.mark_text(
    align="left",
    baseline="middle",
    fontSize=14,
    fontWeight="bold",
).encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("label_anchor:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=None),
    text="path_id:N",
    color=alt.Color("outcome_stroke:N", scale=None, legend=None),
    opacity=alt.condition(path_select, alt.value(1), alt.value(0.38), empty=True),
    tooltip=[
        alt.Tooltip("path:N", title="Path"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
        alt.Tooltip("final_outcome:N", title="Outcome"),
    ],
)

path_text_labels = path_base.mark_text(
    align="left",
    baseline="middle",
    fontSize=14,
    dx=44,
    color="#263238",
).encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("label_anchor:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=None),
    text="path_label:N",
    opacity=alt.condition(path_select, alt.value(1), alt.value(0.38), empty=True),
    tooltip=[
        alt.Tooltip("path:N", title="Path"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
        alt.Tooltip("final_outcome:N", title="Outcome"),
    ],
)

path_count_labels = path_base.mark_text(
    align="left",
    baseline="bottom",
    dx=6,
    dy=-2,
    fontSize=13,
    fontWeight="bold",
    color="#263238",
).encode(
    y=alt.Y("path_id:N", sort=alt.SortField("rank", order="ascending"), axis=None),
    x=alt.X("count:Q", scale=alt.Scale(domain=[-path_label_space, path_chart_max]), axis=None),
    text="count_text:N",
    opacity=alt.condition(path_select, alt.value(1), alt.value(0.38), empty=True),
)

path_chart = (
    path_label_bg
    + path_bars
    + path_id_labels
    + path_text_labels
    + path_count_labels
    + path_click_target
).properties(
    width=640,
    height=255,
    title="Top Ranked Applicant Outcomes",
)

outcome_base = alt.Chart(outcome_counts).transform_filter(filter_expr)

outcome_bars = outcome_base.mark_bar(
    cornerRadiusTopLeft=6,
    cornerRadiusTopRight=6,
    width=84,
    fillOpacity=0.84,
    strokeWidth=2.2,
).encode(
    x=alt.X("outcome:N", title=None),
    y=alt.Y("count:Q", title="Applicants"),
    color=alt.Color("outcome_fill:N", scale=None, legend=None),
    stroke=alt.Color("outcome_stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("outcome_dash:N", scale=dash_scale, legend=None),
    tooltip=[
        alt.Tooltip("outcome:N", title="Outcome"),
        alt.Tooltip("count:Q", title="Applicants", format=","),
    ],
).properties(width=220, height=255, title="Final Outcomes")

outcome_labels = outcome_base.mark_text(
    dy=-8,
    fontSize=14,
    fontWeight="bold",
    color="#263238",
).encode(
    x=alt.X("outcome:N", title=None),
    y=alt.Y("count:Q", title="Applicants"),
    text="count_text:N",
)

selected_outcome_base = alt.Chart(selected_outcome_counts).transform_filter(filter_expr).transform_filter(path_select).transform_joinaggregate(
    selected_path_count="distinct(path)"
).transform_filter(
    "datum.selected_path_count == 1"
)
selected_outcome_bars = selected_outcome_base.mark_bar(
    width=42,
    fillOpacity=0.45,
    strokeWidth=2.6,
).encode(
    x=alt.X("outcome:N", title=None),
    y=alt.Y("count:Q", title="Applicants"),
    color=alt.Color("outcome_fill:N", scale=None, legend=None),
    stroke=alt.Color("outcome_stroke:N", scale=None, legend=None),
    strokeDash=alt.StrokeDash("outcome_dash:N", scale=dash_scale, legend=None),
    tooltip=[
        alt.Tooltip("path:N", title="Selected path"),
        alt.Tooltip("outcome:N", title="Outcome"),
        alt.Tooltip("count:Q", title="Applicants in path", format=","),
        alt.Tooltip("share:Q", title="Share of outcome", format=".1%"),
    ],
)

selected_outcome_pct = selected_outcome_base.mark_text(
    dy=-8,
    fontSize=13,
    fontWeight="bold",
    color="#111827",
).encode(
    x=alt.X("outcome:N", title=None),
    y=alt.Y("count:Q", title="Applicants"),
    text="share_label:N",
)

outcome_chart = outcome_bars + selected_outcome_bars + outcome_labels + selected_outcome_pct

dashboard = alt.vconcat(
    flow_chart,
    alt.hconcat(path_chart, outcome_chart, spacing=22),
    spacing=24,
).add_params(
    year_pick,
    school_pick,
    grade_pick,
).configure_view(
    stroke=None,
).configure_title(
    fontSize=22,
    anchor="start",
).configure_axis(
    labelFontSize=13,
    titleFontSize=14,
)

dashboard
